## 1.Download the datasets from UCSC website

### 1.1 Download the multi-omics data

* Parse the data from the UCSC Xena website in PANCAN cohort:
https://xenabrowser.net/datapages/?cohort=TCGA%20Pan-Cancer%20(PANCAN)&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

<!-- * Copy number (gene-level) - gene-level copy number (gistic2_thresholded)
    * Dataset: https://xenabrowser.net/datapages/?dataset=TCGA.PANCAN.sampleMap%2FGistic2_mutation_Gistic2_all_thresholded.by_genes&host=https%3A%2F%2Ftcga.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443 -->
* Copy number - somatic mutation (SNP and INDEL) - Gene level non-silent mutation
    * Dataset: https://xenabrowser.net/datapages/?dataset=mc3.v0.2.8.PUBLIC.nonsilentGene.xena&host=https%3A%2F%2Fpancanatlas.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

* DNA methylation (Methylation450K)
    * Dataset: https://xenabrowser.net/datapages/?dataset=jhu-usc.edu_PANCAN_HumanMethylation450.betaValue_whitelisted.tsv.synapse_download_5096262.xena&host=https%3A%2F%2Fpancanatlas.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443
    * ID Map: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL16304

* Gene expression RNAseq - TOIL RSEM fpkm
    * Dataset: https://xenabrowser.net/datapages/?dataset=tcga_RSEM_gene_fpkm&host=https%3A%2F%2Ftoil.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443
    
* Protein expression - RPPA
    * Dataset: https://xenabrowser.net/datapages/?dataset=TCGA-RPPA-pancan-clean.xena&host=https%3A%2F%2Fpancanatlas.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

### 1.2 Download the clinical data

### 1.2 Download the clinical data

* Phenotype - Curated clinical data
    * Dataset: https://xenabrowser.net/datapages/?dataset=Survival_SupplementalTable_S1_20171025_xena_sp&host=https%3A%2F%2Fpancanatlas.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

* Phenotype - Immune subtype
    * Dataset: https://xenabrowser.net/datapages/?dataset=Subtype_Immune_Model_Based.txt&host=https%3A%2F%2Fpancanatlas.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

* Phenotype - Molecular subtype
    * Dataset: https://xenabrowser.net/datapages/?dataset=TCGASubtype.20170308.tsv&host=https%3A%2F%2Fpancanatlas.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443

* Phenotype - sample type and primary disease
    * Dataset: https://xenabrowser.net/datapages/?dataset=TCGA_phenotype_denseDataOnlyDownload.tsv&host=https%3A%2F%2Fpancanatlas.xenahubs.net&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443


## 2.Read the files

### 2.1 Read DNA methylation data

In [ ]:
import pandas as pd
methylation_value = pd.read_csv('./UCSC-raw/jhu-usc.edu_PANCAN_HumanMethylation450.betaValue_whitelisted.tsv.synapse_download_5096262.xena', delimiter='\t')

In [ ]:
methylation_value

### 2.2 Read Platform annotations for 450k methylation 

In [ ]:
import pandas as pd
# Reading and processing basic data
try:
    annotation = pd.read_table('./UCSC-raw/GPL16304-47833.txt', delimiter='\t')
    annotation['Distance_closest_TSS'] = annotation['Distance_closest_TSS'].astype(int)
    annotation = annotation[~annotation['Closest_TSS'].apply(lambda x: len(str(x).split(';')) > 1)]
except ValueError as e:
    print(f"Unable to convert 'Closest_TSS' column to integer: {e}")
    problematic_rows = annotation['Distance_closest_TSS'].apply(lambda x: not str(x).isnumeric())
    print("Problematic rows:")
    print(annotation.loc[problematic_rows])

annotation

In [ ]:
map_annotation= annotation[['ID', 'Closest_TSS','Closest_TSS_gene_name', 'Distance_closest_TSS']]
map_annotation

### 2.3 Read mutation data

In [ ]:
# copynumber = pd.read_csv('./UCSC-raw/Gistic2_CopyNumber_Gistic2_all_thresholded.by_genes',sep='\t')
# copynumber
mutation = pd.read_csv('./UCSC-raw/mc3.v0.2.8.PUBLIC.nonsilentGene.xena',sep='\t')
mutation

In [ ]:
original_gene_names = mutation['sample'].copy()

# Use a regular expression to remove the period and any characters following it
mutation['sample'] = mutation['sample'].str.replace(r'\..*', '', regex=True)

# Determine how many rows were changed by comparing the new values to the original ones
rows_changed = (original_gene_names != mutation['sample']).sum()

# Output the updated DataFrame and the number of rows that were changed
mutation, rows_changed

### 2.4 Read gene expression data, mapping information and substitue the the gene name

In [ ]:
gene_expression = pd.read_csv('./UCSC-raw/tcga_RSEM_gene_fpkm', sep='\t')
gene_expression

In [ ]:
gene_expression_map = pd.read_csv('./UCSC-raw/probeMap_gencode.v23.annotation.gene.probemap', sep='\t')

In [ ]:
gene_expression_map

In [ ]:
expression_merged = pd.merge(gene_expression, gene_expression_map, left_on='sample', right_on='id', how='left')

In [ ]:
expression_merged.drop(columns=['sample','chrom', 'chromStart','chromEnd','strand','id'], inplace=True)

In [ ]:
# set gene to the first column
cols = ['gene'] + [col for col in expression_merged if col != 'gene']
gene_expression = expression_merged[cols]

In [ ]:
gene_expression 

In [ ]:
original_gene_names = gene_expression['gene'].copy()

# Use a regular expression to remove the period and any characters following it
gene_expression['gene'] = gene_expression['gene'].str.replace(r'\..*', '', regex=True)

# Determine how many rows were changed by comparing the new values to the original ones
rows_changed = (original_gene_names != gene_expression['gene']).sum()

# Output the updated DataFrame and the number of rows that were changed
gene_expression, rows_changed

### 2.5 Read clinical data

In [ ]:
survival = pd.read_csv('./UCSC-raw/Survival_SupplementalTable_S1_20171025_xena_sp', sep='\t')

In [ ]:
survival

### 2.6 Read immune subtype data

In [ ]:
immune_subtype = pd.read_csv('./UCSC-raw/Subtype_Immune_Model_Based.txt',sep='\t')

In [ ]:
immune_subtype

### 2.7 Read proteomics data

In [ ]:
protein = pd.read_csv('./UCSC-raw/TCGA-RPPA-pancan-clean.xena',sep='\t')

In [ ]:
protein

### 2.8 Read molecular subtype

In [ ]:
cellsub = pd.read_csv('./UCSC-raw/TCGASubtype.20170308.tsv', sep='\t')

In [ ]:
cellsub

### 2.9 Read sample type and primary disease

In [ ]:
dense = pd.read_csv('./UCSC-raw/TCGA_phenotype_denseDataOnlyDownload.tsv', sep='\t')

In [ ]:
dense

## 3.Methylation data process

### 3.1 Define methylation region

In [ ]:
import pandas as pd
import numpy as np

#Define vectorized area determination function for methylation data
def vectorized_determine_region(distances):
    regions = ['Upstream', 'Distal Promoter', 'Proximal Promoter', 'Core Promoter', 'Downstream']
    conditions = [
        (-6000 <= distances) & (distances < -3000),
        (-3000 <= distances) & (distances < -250),
        (-250 <= distances) & (distances < -50),
        (-50 <= distances) & (distances <= 0),
        (0 < distances) & (distances <= 3000)
    ]
    return np.select(conditions, regions, default=None)

### 3.2 Merge the annotation files to the methylation data and apply region function

In [ ]:
# Merging basic data and methylation data
methylation_merged_df = pd.merge(map_annotation, methylation_value, left_on='ID', right_on='sample', how='right')

# Determining the region for each row outside the loop
methylation_merged_df['Region'] = vectorized_determine_region(methylation_merged_df['Distance_closest_TSS'])

methylation_merged_df = methylation_merged_df.dropna(subset=['Region'])  # Remove rows without a region

# Initializing a dictionary to store data for each region
regions_data = {region: pd.DataFrame() for region in ["Upstream", "Distal Promoter", "Proximal Promoter", "Core Promoter", "Downstream"]}

In [ ]:
methylation_merged_df

In [ ]:
# Delete the 'sample' column
methylation_merged_df = methylation_merged_df.drop('sample', axis=1)
# Delete the 'ID' column
methylation_merged_df = methylation_merged_df.drop('ID', axis=1)
# Delete the 'Distance_closest_TSS' column
methylation_merged_df = methylation_merged_df.drop('Distance_closest_TSS', axis=1)


In [ ]:
methylation_merged_df

In [ ]:
original_gene_names = methylation_merged_df['Closest_TSS_gene_name'].copy()

# Use a regular expression to remove the period and any characters following it
methylation_merged_df['Closest_TSS_gene_name'] = methylation_merged_df['Closest_TSS_gene_name'].str.replace(r'\..*', '', regex=True)

# Determine how many rows were changed by comparing the new values to the original ones
rows_changed = (original_gene_names != methylation_merged_df['Closest_TSS_gene_name']).sum()

# Output the updated DataFrame and the number of rows that were changed
methylation_merged_df, rows_changed

In [ ]:
methylation_merged_df['Closest_TSS'] = methylation_merged_df['Closest_TSS'].astype(int)
methylation_merged_df['Closest_TSS_gene_name'] = methylation_merged_df['Closest_TSS_gene_name'].astype(str)
methylation_merged_df['Region'] = methylation_merged_df['Region'].astype(str)

In [ ]:
print(methylation_merged_df[['Closest_TSS', 'Closest_TSS_gene_name', 'Region']].dtypes)

### 3.3 Calculate the average methylation value of five regions

In [ ]:
# find columns not started with 'TCGA'
non_tcga_columns = methylation_merged_df.filter(regex='^(?!TCGA)').columns

print("Columns not starting with TCGA:")
print(non_tcga_columns)

In [ ]:
# obtain all regions
regions = methylation_merged_df['Region'].unique()
regions

In [ ]:
import pandas as pd
# Initialize empty DataFrames for each region
Upstream_df = pd.DataFrame()
Distal_Promoter_df = pd.DataFrame()
Proximal_Promoter_df = pd.DataFrame()
Core_Promoter_df = pd.DataFrame()
Downstream_df = pd.DataFrame()

# Operate on each region
for region in regions:
    # Get all data for this region
    region_data = methylation_merged_df[methylation_merged_df['Region'] == region]
    
    # Group and calculate the average for each (TSS, Region) combination
    grouped = region_data.groupby(['Closest_TSS_gene_name', 'Region'], as_index=False).mean()
    
    # Since we split the data into different files based on Region, we can delete this column
    grouped = grouped.drop(columns=['Region'])
    
    # Print the shape of the grouped data
    print(f"Shape of {region}: {grouped.shape}")
    
    # Assign the grouped data to the respective DataFrame
    if region == 'Upstream':
        Upstream_df = grouped
    elif region == 'Distal Promoter':
        Distal_Promoter_df = grouped
    elif region == 'Proximal Promoter':
        Proximal_Promoter_df = grouped
    elif region == 'Core Promoter':
        Core_Promoter_df = grouped
    elif region == 'Downstream':
        Downstream_df = grouped

    # Optionally, save the data for this region to a new csv file
    # grouped.to_csv(f"{region}_averaged_tss_data.csv", index=False)


### 3.4 Unify gene and TSS for five methylation value files

In [ ]:
import pandas as pd

# from methylation files above DataFrame 
dfs = [Upstream_df, Distal_Promoter_df, Proximal_Promoter_df, Core_Promoter_df, Downstream_df]

# merge those files to find all combos 
all_genes_tss = pd.concat(dfs)['Closest_TSS_gene_name'].drop_duplicates()

In [ ]:
all_genes_tss

In [ ]:
# Merge unique combinations back into each DataFrame and fill NaN values with 0
# Upstream
Upstream_df = pd.merge(all_genes_tss, Upstream_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Upstream_df: {Upstream_df.shape}")

# Distal Promoter
Distal_Promoter_df = pd.merge(all_genes_tss, Distal_Promoter_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Distal_Promoter_df: {Distal_Promoter_df.shape}")

# Proximal Promoter
Proximal_Promoter_df = pd.merge(all_genes_tss, Proximal_Promoter_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Proximal_Promoter_df: {Proximal_Promoter_df.shape}")

# Core Promoter
Core_Promoter_df = pd.merge(all_genes_tss, Core_Promoter_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Core_Promoter_df: {Core_Promoter_df.shape}")

# Downstream
Downstream_df = pd.merge(all_genes_tss, Downstream_df, on=['Closest_TSS_gene_name'], how='outer').fillna(0)
print(f"Shape of Downstream_df: {Downstream_df.shape}")

## 4.Unify genes and patient samples within datasets

### 4.1 Unify gene and TSS for methylation, copynumer, and gene expression data

In [ ]:
mutation.rename(columns={'sample': 'gene_name'}, inplace=True)
mutation = mutation.dropna(subset=['gene_name'])
mutation = mutation.groupby('gene_name', as_index=False).mean()
mutation = mutation.sort_values(by=['gene_name']).reset_index(drop=True)
mutation

In [ ]:
gene_expression.rename(columns={'gene': 'gene_name'}, inplace=True)
gene_expression = gene_expression.dropna(subset=['gene_name'])
gene_expression = gene_expression.groupby('gene_name', as_index=False).mean()
gene_expression = gene_expression.sort_values(by=['gene_name']).reset_index(drop=True)
gene_expression

In [ ]:
protein.rename(columns={'SampleID': 'gene_name'}, inplace=True)
protein = protein.dropna(subset=['gene_name'])
protein = protein.groupby('gene_name', as_index=False).mean()
protein = protein.sort_values(by=['gene_name']).reset_index(drop=True)
protein

In [ ]:
# Upstream
Upstream_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Upstream_df = Upstream_df.dropna(subset=['gene_name'])
Upstream_df = Upstream_df.groupby('gene_name', as_index=False).mean()
Upstream_df = Upstream_df.sort_values(by=['gene_name']).reset_index(drop=True)
# Find indices where 'row_name' is 'unknown'
Upstream_indices_to_drop = Upstream_df[Upstream_df['gene_name'] == 'unknown'].index
Upstream_df.drop(Upstream_indices_to_drop, inplace=True)

# Distal Promoter
Distal_Promoter_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Distal_Promoter_df = Distal_Promoter_df.dropna(subset=['gene_name'])
Distal_Promoter_df = Distal_Promoter_df.groupby('gene_name', as_index=False).mean()
Distal_Promoter_df = Distal_Promoter_df.sort_values(by=['gene_name']).reset_index(drop=True)
# Find indices where 'row_name' is 'unknown'
Distal_Promoter_indices_to_drop = Distal_Promoter_df[Distal_Promoter_df['gene_name'] == 'unknown'].index
Distal_Promoter_df.drop(Distal_Promoter_indices_to_drop, inplace=True)

# Proximal Promoter
Proximal_Promoter_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Proximal_Promoter_df = Proximal_Promoter_df.dropna(subset=['gene_name'])
Proximal_Promoter_df = Proximal_Promoter_df.groupby('gene_name', as_index=False).mean()
Proximal_Promoter_df = Proximal_Promoter_df.sort_values(by=['gene_name']).reset_index(drop=True)
# Find indices where 'row_name' is 'unknown'
Proximal_Promoter_indices_to_drop = Proximal_Promoter_df[Proximal_Promoter_df['gene_name'] == 'unknown'].index
Proximal_Promoter_df.drop(Proximal_Promoter_indices_to_drop, inplace=True)

# Core Promoter
Core_Promoter_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Core_Promoter_df = Core_Promoter_df.dropna(subset=['gene_name'])
Core_Promoter_df = Core_Promoter_df.groupby('gene_name', as_index=False).mean()
Core_Promoter_df = Core_Promoter_df.sort_values(by=['gene_name']).reset_index(drop=True)
# Find indices where 'row_name' is 'unknown'
Core_Promoter_indices_to_drop = Core_Promoter_df[Core_Promoter_df['gene_name'] == 'unknown'].index
Core_Promoter_df.drop(Core_Promoter_indices_to_drop, inplace=True)

# Downstream
Downstream_df.rename(columns={'Closest_TSS_gene_name': 'gene_name'}, inplace=True)
Downstream_df = Downstream_df.dropna(subset=['gene_name'])
Downstream_df = Downstream_df.groupby('gene_name', as_index=False).mean()
Downstream_df = Downstream_df.sort_values(by=['gene_name']).reset_index(drop=True)
# Find indices where 'row_name' is 'unknown'
Downstream_indices_to_drop = Downstream_df[Downstream_df['gene_name'] == 'unknown'].index
Downstream_df.drop(Downstream_indices_to_drop, inplace=True)

display(Upstream_df)
display(Distal_Promoter_df)
display(Proximal_Promoter_df)
display(Core_Promoter_df)
display(Downstream_df)

In [ ]:
ensembl_data_unique_gene = pd.read_csv("./UCSC-raw/mart_export_unique_gene.txt")
ensembl_data_unique_gene

In [ ]:
#filter the gene not in ensembl
mutation = mutation[mutation['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]
gene_expression = gene_expression[gene_expression['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]
Upstream_df = Upstream_df[Upstream_df['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]
protein = protein[protein['gene_name'].isin(ensembl_data_unique_gene['Gene name'])]

In [ ]:
import pandas as pd

# Convert the gene name columns from each DataFrame to sets
mutation_genes = set(mutation['gene_name'])
print(f"Number of mutation genes: {len(mutation_genes)}")
gene_expression_genes = set(gene_expression['gene_name'])
print(f"Number of gene expression genes: {len(gene_expression_genes)}")
methylation_genes = set(Upstream_df['gene_name'])
print(f"Number of methylation genes: {len(methylation_genes)}")
protein_genes = set(protein['gene_name'])
print(f"Number of protein genes: {len(protein_genes)}")
# Find the intersection of the three sets
common_genes = mutation_genes | gene_expression_genes | methylation_genes | protein_genes

# Convert the intersection back to a list, if needed
common_genes_list = list(common_genes)

# Print the number of common genes
print(f"Number of common genes: {len(common_genes)}")

In [ ]:
#count the Transcript type
ensembl_data_type = pd.read_csv("./UCSC-raw/mart_export.txt")
ensembl_data_type= ensembl_data_type.rename(columns={'Gene name': 'gene_name'})
ensembl_data_type = ensembl_data_type.drop_duplicates(subset='gene_name', keep='first')
ensembl_data_type
# Now, merge the two dataframes on the 'gene_name' column
Upstream_df_transcript = pd.merge(Upstream_df, ensembl_data_type, on='gene_name', how='left')
protein_coding_count = (Upstream_df_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

protein_transcript = pd.merge(protein, ensembl_data_type, on='gene_name', how='left')
protein_coding_count = (protein_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

mutation_transcript = pd.merge(mutation, ensembl_data_type, on='gene_name', how='left')
protein_coding_count = (mutation_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

gene_expression_transcript = pd.merge(gene_expression, ensembl_data_type, on='gene_name', how='left')
protein_coding_count = (gene_expression_transcript['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

#### 4.1.1 Make the gene expression, mutation, copy number and proteomics data inputted

In [ ]:
common_genes_df = pd.DataFrame(common_genes, columns=['gene_name'])
common_genes_df = common_genes_df.sort_values(by=['gene_name']).reset_index(drop=True)
common_genes_df

In [ ]:
merged_data_transcript_Upstream_df = pd.merge(common_genes_df, ensembl_data_type, on='gene_name', how='left')
protein_coding_count = (merged_data_transcript_Upstream_df['Transcript type'] == 'protein_coding').sum()
print(f'Number of rows with "protein_coding": {protein_coding_count}')

In [ ]:
unique_values = merged_data_transcript_Upstream_df['Transcript type'].value_counts()
unique_values

In [ ]:
gene_expression_inputted = pd.merge(common_genes_df ,gene_expression,on='gene_name',how='outer').fillna(0)
protein_inputted = pd.merge(common_genes_df, protein, on='gene_name', how='outer').fillna(0)

Upstream_df_inputted = pd.merge(common_genes_df, Upstream_df,on='gene_name',how='outer').fillna(0)
Distal_Promoter_df_inputted = pd.merge(common_genes_df, Distal_Promoter_df,on='gene_name',how='outer').fillna(0)
Proximal_Promoter_df_inputted = pd.merge(common_genes_df, Proximal_Promoter_df,on='gene_name',how='outer').fillna(0)
Core_Promoter_df_inputted = pd.merge(common_genes_df, Core_Promoter_df,on='gene_name',how='outer').fillna(0)
Downstream_df_inputted = pd.merge(common_genes_df, Downstream_df,on='gene_name',how='outer').fillna(0)

mutation_inputted = pd.merge(common_genes_df, mutation, on='gene_name', how='outer').fillna(-1)

display(gene_expression_inputted)

#### 4.1.2 Intersecting genes with various databases

In [ ]:
import pandas as pd
# Add the gene names from databases like [KEGG / BioGRID] to intersect with the common genes
# KEGG
kegg_pathway_df = pd.read_csv('./data/kg_data/KEGG/full_kegg_pathway_list.csv')
kegg_pathway_df = kegg_pathway_df[['source', 'target', 'pathway_name']]
kegg_df = kegg_pathway_df[kegg_pathway_df['pathway_name'].str.contains('signaling pathway|signaling pathways', case=False)]
print(kegg_df['pathway_name'].value_counts())
kegg_df = kegg_df.rename(columns={'source': 'src', 'target': 'dest'})
src_list = list(kegg_df['src'])
dest_list = list(kegg_df['dest'])
path_list = list(kegg_df['pathway_name'])
# ADJUST ALL GENES TO UPPERCASE
up_src_list = []
for src in src_list:
    up_src = src.upper()
    up_src_list.append(up_src)
up_dest_list = []
for dest in dest_list:
    up_dest = dest.upper()
    up_dest_list.append(up_dest)
up_kegg_conn_dict = {'src': up_src_list, 'dest': up_dest_list}
up_kegg_df = pd.DataFrame(up_kegg_conn_dict)
up_kegg_df = up_kegg_df.drop_duplicates()
up_kegg_df.to_csv('./data/kg_data/KEGG/up_kegg.csv', index=False, header=True)
kegg_gene_list = list(set(list(up_kegg_df['src']) + list(up_kegg_df['dest'])))
print('----- NUMBER OF GENES IN KEGG: ' + str(len(kegg_gene_list)) + ' -----')
print(up_kegg_df.shape)

up_kegg_path_conn_dict = {'src': up_src_list, 'dest': up_dest_list, 'path': path_list}
up_kegg_path_df = pd.DataFrame(up_kegg_path_conn_dict)
up_kegg_path_df = up_kegg_path_df.drop_duplicates()
up_kegg_path_df.to_csv('./data/kg_data/KEGG/up_kegg_path.csv', index=False, header=True)
kegg_path_gene_list = list(set(list(up_kegg_path_df['src']) + list(up_kegg_path_df['dest'])))
print('----- NUMBER OF GENES IN KEGG PATH: ' + str(len(kegg_path_gene_list)) + ' -----')
print(up_kegg_path_df.shape)

In [ ]:
# BioGRID
biogrid_df = pd.read_table('./data/kg_data/BioGrid/BIOGRID-ALL-3.5.174.mitab.Symbol.txt', delimiter = '\t')
eh_list = list(biogrid_df['e_h'])
et_list = list(biogrid_df['e_t'])
# ADJUST ALL GENES TO UPPERCASE
up_eh_list = []
for eh in eh_list:
    up_eh = eh.upper()
    up_eh_list.append(up_eh)
up_et_list = []
for et in et_list:
    up_et = et.upper()
    up_et_list.append(up_et)
up_biogrid_conn_dict = {'src': up_eh_list, 'dest': up_et_list}
up_biogrid_df = pd.DataFrame(up_biogrid_conn_dict)
print(up_biogrid_df)
print(up_biogrid_df.shape)
up_biogrid_df.to_csv('./data/kg_data/BioGrid/up_biogrid.csv', index = False, header = True)
up_biogrid_gene_list = list(set(list(up_biogrid_df['src']) + list(up_biogrid_df['dest'])))
print('----- NUMBER OF GENES IN BioGRID: ' + str(len(up_biogrid_gene_list)) + ' -----')

In [ ]:
# STRING
string_df = pd.read_csv('./data/kg_data/STRING/9606.protein.links.detailed.v11.0_sym.csv', low_memory=False)
src_list = list(string_df['Source'])
tar_list = list(string_df['Target'])
# ADJUST ALL GENES TO UPPERCASE
up_src_list = []
for src in src_list:
    up_src = src.upper()
    up_src_list.append(up_src)
up_tar_list = []
for tar in tar_list:
    up_tar = tar.upper()
    up_tar_list.append(up_tar)
up_string_conn_dict = {'src': up_src_list, 'dest': up_tar_list}
up_string_df = pd.DataFrame(up_string_conn_dict)
print(up_string_df)
up_string_df.to_csv('./data/kg_data/STRING/up_string.csv', index = False, header = True)
up_string_gene_list = list(set(list(up_string_df['src']) + list(up_string_df['dest'])))
print('----- NUMBER OF GENES IN STRING: ' + str(len(up_string_gene_list)) + ' -----')

In [ ]:
# intersect the [common genes] with the genes in the different databases [KEGG / BioGRID / STRING]
selected_database = 'KEGG'
# selected_database = 'BioGRID'
# selected_database = 'STRING'
if selected_database == 'KEGG':
    edge_common_genes = list(set(common_genes) & set(kegg_gene_list))
    print('----- NUMBER OF INTERSECTED GENES IN KEGG: ' + str(len(edge_common_genes)) + ' -----')
elif selected_database == 'BioGRID':
    edge_common_genes = list(set(common_genes) & set(up_biogrid_gene_list))
    print('----- NUMBER OF INTERSECTED GENES IN BioGRID: ' + str(len(edge_common_genes)) + ' -----')
elif selected_database == 'STRING':
    edge_common_genes = list(set(common_genes) & set(up_string_gene_list))
    print('----- NUMBER OF INTERSECTED GENES IN STRING: ' + str(len(edge_common_genes)) + ' -----')

# filter the genes in the different databases [KEGG / BioGRID / STRING] with the [common genes]
if selected_database == 'KEGG':
    filtered_up_kegg_df = up_kegg_df[up_kegg_df['src'].isin(edge_common_genes) & up_kegg_df['dest'].isin(edge_common_genes)]
    src_list = list(filtered_up_kegg_df['src'])
    dest_list = list(filtered_up_kegg_df['dest'])
    all_list = sorted(list(set(src_list + dest_list)))
    print('----- NUMBER OF INTERSECTED GENES IN KEGG: ' + str(len(all_list)) + ' -----')
    edge_common_genes = all_list
    filtered_up_kegg_df = filtered_up_kegg_df.drop_duplicates()
    filtered_up_kegg_df = filtered_up_kegg_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW KEGG EDGE CONNECTIONS: ' + str(len(filtered_up_kegg_df)) + ' -----')
    filtered_up_kegg_path_df = up_kegg_path_df[up_kegg_path_df['src'].isin(edge_common_genes) & up_kegg_path_df['dest'].isin(edge_common_genes)]
    filtered_up_kegg_path_df = filtered_up_kegg_path_df.drop_duplicates()
    filtered_up_kegg_path_df = filtered_up_kegg_path_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW KEGG PATHWAY CONNECTIONS: ' + str(len(filtered_up_kegg_path_df)) + ' -----')
elif selected_database == 'BioGRID':
    filtered_up_biogrid_df = up_biogrid_df[up_biogrid_df['src'].isin(edge_common_genes) & up_biogrid_df['dest'].isin(edge_common_genes)]
    filtered_up_biogrid_df = filtered_up_biogrid_df.drop_duplicates()
    filtered_up_biogrid_df = filtered_up_biogrid_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW BioGRID EDGE CONNECTIONS: ' + str(len(filtered_up_biogrid_df)) + ' -----')
elif selected_database == 'STRING':
    filtered_up_string_df = up_string_df[up_string_df['src'].isin(edge_common_genes) & up_string_df['dest'].isin(edge_common_genes)]
    filtered_up_string_df = filtered_up_string_df.drop_duplicates()
    filtered_up_string_df = filtered_up_string_df.sort_values(by=['src', 'dest']).reset_index(drop=True)
    print('----- NEW STRING EDGE CONNECTIONS: ' + str(len(filtered_up_string_df)) + ' -----')

In [ ]:
if selected_database == 'KEGG':
    display(filtered_up_kegg_df)
    display(filtered_up_kegg_path_df)
elif selected_database == 'BioGRID':
    display(filtered_up_biogrid_df)
elif selected_database == 'STRING':
    display(filtered_up_string_df)

#### 4.1.3 Filtering the gene names across the gene expression, cnv, proteomics and methylation

In [ ]:
# select common genes in mutation data
mutation_filtered = mutation_inputted.loc[mutation_inputted['gene_name'].isin(edge_common_genes)]
mutation_filtered = mutation_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
# select common genes in gene expression data
gene_expression_filtered = gene_expression_inputted.loc[gene_expression_inputted['gene_name'].isin(edge_common_genes)]
gene_expression_filtered = gene_expression_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
# select common genes in protein data
protein_filtered = protein_inputted.loc[protein_inputted['gene_name'].isin(edge_common_genes)]
protein_filtered = protein_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
# select common genes in upstream data
Upstream_df_filtered = Upstream_df_inputted.loc[Upstream_df_inputted['gene_name'].isin(edge_common_genes)]
Upstream_df_filtered = Upstream_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
# select common genes in distal promoter data
Distal_Promoter_df_filtered = Distal_Promoter_df_inputted.loc[Distal_Promoter_df_inputted['gene_name'].isin(edge_common_genes)]
Distal_Promoter_df_filtered = Distal_Promoter_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
# select common genes in proximal promoter data
Proximal_Promoter_df_filtered = Proximal_Promoter_df_inputted.loc[Proximal_Promoter_df_inputted['gene_name'].isin(edge_common_genes)]
Proximal_Promoter_df_filtered = Proximal_Promoter_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
# select common genes in core promoter data
Core_Promoter_df_filtered = Core_Promoter_df_inputted.loc[Core_Promoter_df_inputted['gene_name'].isin(edge_common_genes)]
Core_Promoter_df_filtered = Core_Promoter_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)
# select common genes in downstream data
Downstream_df_filtered = Downstream_df_inputted.loc[Downstream_df_inputted['gene_name'].isin(edge_common_genes)]
Downstream_df_filtered = Downstream_df_filtered.sort_values(by=['gene_name']).reset_index(drop=True)

In [ ]:
mutation_filtered

### 4.2 Unify patient samples within methylation, copynumer,  gene expression, clinical, proteomics, molecular subtype, and sample type, primary disease datasets

In [ ]:
# clinical data
survival

In [ ]:
# immune subtype
immune_subtype

In [ ]:
# proteomics data
protein_filtered

In [ ]:
# molecular subtype
cellsub.rename(columns={'sampleID': 'sample'}, inplace=True)
cellsub

In [ ]:
# sample type and primary disease
dense

In [ ]:
# Upstream_df_filtered, Distal_Promoter_df_filtered, Proximal_Promoter_df_filtered, Core_Promoter_df_filtered, Downstream_df_filtered
# mutation_filtered, gene_expression_filtered
# survival, protein, cellsub, dense

# Extract column names starting with 'TCGA' from methylation datasets
tcga_columns_upstream = [col for col in Upstream_df_filtered.columns if col.startswith('TCGA')]
tcga_columns_distal = [col for col in Distal_Promoter_df_filtered.columns if col.startswith('TCGA')]
tcga_columns_proximal = [col for col in Proximal_Promoter_df_filtered.columns if col.startswith('TCGA')]
tcga_columns_core = [col for col in Core_Promoter_df_filtered.columns if col.startswith('TCGA')]
tcga_columns_downstream = [col for col in Downstream_df_filtered.columns if col.startswith('TCGA')]

# Extract 'TCGA' columns from other datasets
tcga_columns_mutation = [col for col in mutation_filtered.columns if col.startswith('TCGA')]
tcga_columns_gene_expression = [col for col in gene_expression_filtered.columns if col.startswith('TCGA')]
tcga_columns_survival = [col for col in survival['sample'] if col.startswith('TCGA')]
tcga_columns_protein = [col for col in protein_filtered.columns if col.startswith('TCGA')]
tcga_columns_cellsub = [col for col in cellsub['sample'] if col.startswith('TCGA')]
tcga_columns_dense = [col for col in dense['sample'] if col.startswith('TCGA')]
tcga_columns_immune_subtype = [col for col in immune_subtype['sample'] if col.startswith('TCGA')]
# Find the intersection of TCGA column names across all DataFrames
common_tcga_columns = set(tcga_columns_upstream) & set(tcga_columns_distal) & set(tcga_columns_proximal) & set(tcga_columns_core) & set(tcga_columns_downstream) & set(tcga_columns_mutation) & set(tcga_columns_gene_expression) & set(tcga_columns_survival) & set(tcga_columns_protein) & set(tcga_columns_cellsub) & set(tcga_columns_dense) & set(tcga_columns_immune_subtype)

# Convert the intersection back to a list, if needed
common_tcga_columns_list = sorted(list(common_tcga_columns))

# Print the number and the list of common TCGA columns
print(f"Number of common TCGA columns: {len(common_tcga_columns)}")

In [ ]:
print(tcga_columns_upstream)
print(f"Methylation Dataset has: {len(tcga_columns_upstream)}")
tcga_columns_methylation_withsurvival = set(tcga_columns_upstream) & set(tcga_columns_survival) & set(tcga_columns_cellsub) & set(tcga_columns_dense) & set(tcga_columns_immune_subtype)
print(f"After intersection with survival: {len(tcga_columns_methylation_withsurvival)}")

print(f"Mutation Dataset has: {len(tcga_columns_mutation)}")
tcga_columns_mutation_withsurvival = set(tcga_columns_mutation) & set(tcga_columns_survival) & set(tcga_columns_cellsub) & set(tcga_columns_dense) & set(tcga_columns_immune_subtype)
print(f"After intersection with survival: {len(tcga_columns_mutation_withsurvival)}")

print(f"RNASeq Dataset has: {len(tcga_columns_gene_expression)}")
tcga_columns_gene_expression_withsurvival = set(tcga_columns_gene_expression) & set(tcga_columns_survival) & set(tcga_columns_cellsub) & set(tcga_columns_dense) & set(tcga_columns_immune_subtype)
print(f"After intersection with survival: {len(tcga_columns_gene_expression_withsurvival)}")

print(f"Protein Expression Dataset has: {len(tcga_columns_protein)}")
tcga_columns_protein_withsurvival = set(tcga_columns_protein) & set(tcga_columns_survival) & set(tcga_columns_cellsub) & set(tcga_columns_dense) & set(tcga_columns_immune_subtype)
print(f"After intersection with survival: {len(tcga_columns_protein_withsurvival)}")
# R_columns_cnv, 
# R_columns_gene_expression, 
# R_columns_survival, 
# R_columns_protein

In [ ]:
import venn
labels = venn.get_labels([
    tcga_columns_mutation_withsurvival,
    tcga_columns_gene_expression_withsurvival,
    tcga_columns_protein_withsurvival,
    tcga_columns_methylation_withsurvival
], fill=['number'])
fig, ax = venn.venn4(labels, names=['Mutation Dataset', 'RNASeq Dataset', 'Protein Expression Dataset','Methylation Dataset'])
fig.show()


In [ ]:
tcga_columns_mutation_withsurvival
tcga_columns_gene_expression_withsurvival
tcga_columns_protein_withsurvival
tcga_columns_methylation_withsurvival

intersections = {
    'Mutation∩RNASeq': tcga_columns_mutation_withsurvival & tcga_columns_gene_expression_withsurvival,
    'Mutation∩Protein Expression': tcga_columns_mutation_withsurvival & tcga_columns_protein_withsurvival,
    'Mutation∩Methylation': tcga_columns_mutation_withsurvival & tcga_columns_methylation_withsurvival,
    'RNASeq∩Protein Expression': tcga_columns_gene_expression_withsurvival & tcga_columns_protein_withsurvival,
    'RNASeq∩Methylation': tcga_columns_gene_expression_withsurvival & tcga_columns_methylation_withsurvival,
    'Protein Expression∩Methylation': tcga_columns_protein_withsurvival & tcga_columns_methylation_withsurvival,
    'Mutation∩RNASeq∩Protein Expression': tcga_columns_mutation_withsurvival & tcga_columns_gene_expression_withsurvival & tcga_columns_protein_withsurvival,
    'Mutation∩RNASeq∩Methylation': tcga_columns_mutation_withsurvival & tcga_columns_gene_expression_withsurvival & tcga_columns_methylation_withsurvival,
    'Mutation∩Protein Expression∩Methylation': tcga_columns_mutation_withsurvival & tcga_columns_protein_withsurvival & tcga_columns_methylation_withsurvival,
    'RNASeq∩Protein Expression∩Methylation': tcga_columns_gene_expression_withsurvival & tcga_columns_protein_withsurvival & tcga_columns_methylation_withsurvival,
}

for name, intersection in intersections.items():
    print(f"{name} = {len(intersection)}")


In [ ]:
from upsetplot import plot
import pandas as pd
import matplotlib.pyplot as plt
# Sample data: replace these with your actual sets
sets = {
    'Mutation Dataset': tcga_columns_mutation_withsurvival,
    'RNASeq Dataset': tcga_columns_gene_expression_withsurvival,
    'Protein Expression Dataset': tcga_columns_protein_withsurvival,
    'Methylation Dataset': tcga_columns_methylation_withsurvival
}

# Identify all unique elements across all sets
all_elements = set.union(*sets.values())

# Transform the data for UpSetPlot
data = []
for element in all_elements:
    data.append([
        element in sets['Mutation Dataset'],
        element in sets['RNASeq Dataset'],
        element in sets['Protein Expression Dataset'],
        element in sets['Methylation Dataset']
    ])

# Create a DataFrame
df = pd.DataFrame(data, columns=['Mutation Dataset', 'RNASeq Dataset', 'Protein Expression Dataset', 'Methylation Dataset'])

# Convert boolean values to integers
df = df.astype(int)

# Prepare the DataFrame for UpSetPlot
df_upset = df.groupby(list(df.columns)).size()

# Plot using UpSetPlot
# plot(df_upset, orientation="vertical",show_counts=True)

plt.figure(figsize=(16, 6))
upset_axes = plot(df_upset, orientation="vertical", show_counts=True,element_size=40)
# totals_ax = upset_axes['totals']
# totals_ax.tick_params(labelleft=False, left=False)
# totals_ax.tick_params(bottom=False, labelbottom=False)
# for spine in totals_ax.spines.values():
#     spine.set_visible(False)
# plt.show()

In [ ]:
# Define columns to keep along with common TCGA columns
additional_cols_methylation = ['gene_name']

# Filter each methylation DataFrame
Upstream_df_filtered = Upstream_df_filtered[additional_cols_methylation + common_tcga_columns_list]
Distal_Promoter_df_filtered = Distal_Promoter_df_filtered[additional_cols_methylation + common_tcga_columns_list]
Proximal_Promoter_df_filtered = Proximal_Promoter_df_filtered[additional_cols_methylation + common_tcga_columns_list]
Core_Promoter_df_filtered = Core_Promoter_df_filtered[additional_cols_methylation + common_tcga_columns_list]
Downstream_df_filtered = Downstream_df_filtered[additional_cols_methylation + common_tcga_columns_list]


In [ ]:
Upstream_df_filtered

In [ ]:
# Define columns to keep along with common TCGA columns
additional_cols_mutation = ['gene_name']

# Filter the mutation DataFrame
mutation_filtered = mutation_filtered[additional_cols_mutation + common_tcga_columns_list]
mutation_filtered.columns.values[0] = 'gene_name'
mutation_filtered.sort_values(by='gene_name', inplace=True)
mutation_filtered

In [ ]:
# Define columns to keep along with common TCGA columns
additional_cols_gene_expression = ['gene_name']

# Filter the gene expression DataFrame
gene_expression_filtered = gene_expression_filtered[additional_cols_gene_expression + common_tcga_columns_list]
gene_expression_filtered

In [ ]:
# Define columns to keep along with common TCGA columns
additional_cols_protein = ['gene_name']

# Filter the protein DataFrame
protein_filtered = protein_filtered[additional_cols_protein + common_tcga_columns_list]
protein_filtered

In [ ]:
# Filter rows based on common TCGA identifiers
immune_subtype_filtered = immune_subtype[immune_subtype['sample'].isin(common_tcga_columns_list)]
immune_subtype_filtered = immune_subtype_filtered.sort_values(by=['sample']).reset_index(drop=True)

survival_filtered = survival[survival['sample'].isin(common_tcga_columns_list)]
survival_filtered = survival_filtered.sort_values(by=['sample']).reset_index(drop=True)

cellsub.rename(columns={'sampleID': 'sample'}, inplace=True)
cellsub_filtered = cellsub[cellsub['sample'].isin(common_tcga_columns_list)]
cellsub_filtered = cellsub_filtered.sort_values(by=['sample']).reset_index(drop=True)

dense_filtered = dense[dense['sample'].isin(common_tcga_columns_list)]
dense_filtered = dense_filtered.sort_values(by=['sample']).reset_index(drop=True)

In [ ]:
immune_subtype_filtered

In [ ]:
survival_filtered

In [ ]:
cellsub_filtered

In [ ]:
dense_filtered

### 4.3 proteomics missing value and intersection


In [ ]:
protein_filtered

In [ ]:
#calculate the NaN proportion of each row
nan_proportions = protein_filtered.isna().mean(axis=1)
# Display the results
print(nan_proportions)
display(protein_filtered)

## 5.Gene name/patient samples/ pheotype file lists

### 5.1 gene name and patient samples lists

In [ ]:
gene_list = gene_expression_filtered['gene_name']
gene_list

In [ ]:
protein_list = protein_filtered['gene_name'].tolist()
print(len(protein_list))
protein_list

In [ ]:
intersection = list(set(gene_list) & set(protein_list))
len(intersection)

In [ ]:
patient_sample_list = pd.DataFrame(common_tcga_columns_list,columns=['sample'])
patient_sample_list

### 5.2 phenotype lists

In [ ]:
immune_subtype_filtered

In [ ]:
survival_filtered

In [ ]:
cellsub_filtered

In [ ]:
import pandas as pd

# extract phenotype names
immune_phenotypes = immune_subtype_filtered.columns[1:].tolist()
survival_phenotypes = survival_filtered.columns[2:].tolist() # _PATIENT infor is not needed (sample id)
dense_phenotypes = dense_filtered.columns[2:].tolist() # sample_type_id infor is not needed (all = 1)
cellsub_phenotypes = cellsub_filtered.columns[1:].tolist()

# creat phenotype name and source
phenotype_list = []
phenotype_list.extend([(p, 'immunesub') for p in immune_phenotypes])
phenotype_list.extend([(p, 'survival') for p in survival_phenotypes])
phenotype_list.extend([(p, 'dense') for p in dense_phenotypes])
phenotype_list.extend([(p, 'cellsub') for p in cellsub_phenotypes])

# list DataFrame
phenotype_lists = pd.DataFrame(phenotype_list, columns=['Phenotype_Name', 'Phenotype_Source'])
phenotype_lists

# 6 Statistical Analysis on Samples

In [ ]:
survival_filtered.drop(columns=['_PATIENT'], inplace=True)
OS_list = survival_filtered[survival_filtered['OS'] == 1]['sample'].tolist()
nonOS_list = survival_filtered[survival_filtered['OS'] == 0]['sample'].tolist()

## 6.1 Statistical analysis for clinical features

In [ ]:
OS_phenodata_df = survival_filtered[survival_filtered['sample'].isin(OS_list)].reset_index(drop=True)
nonOS_phenodata_df = survival_filtered[survival_filtered['sample'].isin(nonOS_list)].reset_index(drop=True)
display(OS_phenodata_df)
display(nonOS_phenodata_df)
from scipy.stats import ks_2samp
from scipy.stats import mannwhitneyu
pvalue_ks_OS_list = []
invalid_columns = []

label_phenodata_df_col_name_list = survival_filtered.select_dtypes(include=['number']).drop(columns=['OS']).columns.tolist()

for col_name in label_phenodata_df_col_name_list:
    OS_feature_list = OS_phenodata_df[col_name].dropna().tolist()
    nonOS_feature_list = nonOS_phenodata_df[col_name].dropna().tolist()
    
    if OS_feature_list and nonOS_feature_list:  # Ensure the lists are not empty
        ks_stat, p_value_ks = ks_2samp(OS_feature_list, nonOS_feature_list)
        pvalue_ks_OS_list.append(p_value_ks)
    else:
        invalid_columns.append(col_name)  # Track invalid columns
        print(f"{col_name} column cannot be performed p-value")
        
# Remove the invalid columns from the lists
label_phenodata_df_col_name_list = [col for col in label_phenodata_df_col_name_list if col not in invalid_columns]
pvalue_ks_OS_list = [pval for col, pval in zip(label_phenodata_df_col_name_list, pvalue_ks_OS_list) if col not in invalid_columns]


print(len(label_phenodata_df_col_name_list))
print(len(pvalue_ks_OS_list))

label_phenodata_col_name_pvalue_df = pd.DataFrame({
    'features': label_phenodata_df_col_name_list,
    'pvalue': pvalue_ks_OS_list,
})
import os
if os.path.exists('./data_UCSC') == False:
    os.mkdir('./data_UCSC/')
if os.path.exists('./data_UCSC/stat_data/') == False:
    os.mkdir('./data_UCSC/stat_data/')
label_phenodata_col_name_pvalue_df.to_csv('./data_UCSC/stat_data/label_phenodata_col_name_pvalue_df.csv', index=False, header=True)

In [ ]:
OS_phenodata_average_list = OS_phenodata_df[label_phenodata_df_col_name_list].mean().tolist()
nonOS_phenodata_average_list = nonOS_phenodata_df[label_phenodata_df_col_name_list].mean().tolist()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.colors import TwoSlopeNorm
import pandas as pd

cmap = 'Oranges_r'
# cmap = 'Oranges'
list1 = pvalue_ks_OS_list


#retrieve unique labels
ylabels = label_phenodata_df_col_name_list
print(len(ylabels))
xlabels = ['OS vs nonOS']
ylabels_num_list = list(np.arange(0, len(ylabels))) 
xlabels_num_list = len(ylabels) * [0] 
xn = len(xlabels)
yn = len(ylabels)
#retrieve size and color information    
s = np.array(list1 )
c = np.array(list1)

#preparation of the figure with its grid
fig, ax = plt.subplots(figsize=(20, 10))
ax.set_xlim(-0.5, xn-0.5)
ax.set_ylim(-0.5, yn-0.5)
ax.set(xticks=np.arange(xn), yticks=np.arange(yn),
       xticklabels=xlabels, yticklabels=ylabels)

ax.set_xticks(np.arange(xn)-0.5, minor=True)
ax.set_yticks(np.arange(yn)-0.5, minor=True)
# Rotate x-axis labels
plt.xticks(rotation=45, ha='right', fontsize=8)
# plt.xticks(rotation=90)
ax.grid(which='minor')
#ensure circles are displayed as circles
ax.set_aspect("equal", "box")

#create circles patches and colorbar
# R = 0.4 - s/s.max()/2
R = [0.3] * len(s)
# R = 0.3-s/(s.max()/0.3)
circles = [plt.Circle((xlabels_num_list[i], ylabels_num_list[i]), radius=r) for i, r in enumerate(R)]
norm = TwoSlopeNorm(vmin=0, vmax=1, vcenter=0.1)
col = PatchCollection(circles, array=c, cmap=cmap, norm=norm)
ax.add_collection(col)
fig.colorbar(col, shrink=0.2, aspect=10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sample data for three bar plots
x1 = np.array(OS_phenodata_average_list)
x2 = np.array(nonOS_phenodata_average_list)
y1 = np.array(label_phenodata_df_col_name_list)

# Create a figure and subplots with shared y-axis
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 10), sharey=True)

# Set colors for each bar plot
colors = ['C0', 'C1']

# Plot the first bar plot
axes[0].barh(y1, x1, color=colors[0])
axes[0].set_title('T2ds bar plot')

# Plot the second bar plot
axes[1].barh(y1, x2, color=colors[1])
axes[1].set_title('Pre_T2ds bar plot')


# Set common labels and title for the subplots
fig.text(0.5, -0.04, 'Average Values for Each Type of Patient', ha='center')
fig.text(-0.01, 0.5, 'Visit 1 Features', va='center', rotation='vertical')
fig.suptitle('Three Classifications of Patients Bar Plots')

# Adjust the spacing between subplots
plt.tight_layout()

## 6.2 Statistical analysis with Fold-Change for clinical features

## 6.3 Statistical analysis for transcriptomics data

In [ ]:
# Keep the subject in the columns for certain types of patients
OS_gene_expression_df = gene_expression_filtered[['gene_name'] + OS_list]
display(OS_gene_expression_df)
OS_gene_expression_transposed_df = OS_gene_expression_df.T
OS_gene_expression_transposed_df.columns = OS_gene_expression_transposed_df.iloc[0]
OS_gene_expression_transposed_df = OS_gene_expression_transposed_df[1:] 
# convert index to first column with the name 'subject' and remove the index name
OS_gene_expression_transposed_df.reset_index(level=0, inplace=True)
OS_gene_expression_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
display(OS_gene_expression_transposed_df)

nonOS_gene_expression_df = gene_expression_filtered[['gene_name'] + nonOS_list] 
display(nonOS_gene_expression_df)
nonOS_gene_expression_transposed_df = nonOS_gene_expression_df.T
nonOS_gene_expression_transposed_df.columns = nonOS_gene_expression_transposed_df.iloc[0]
nonOS_gene_expression_transposed_df = nonOS_gene_expression_transposed_df[1:]
nonOS_gene_expression_transposed_df.reset_index(level=0, inplace=True)
nonOS_gene_expression_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
display(nonOS_gene_expression_transposed_df)

In [ ]:
from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

gene_expression_col_name_list = list(OS_gene_expression_transposed_df.columns)[1:]
for col_name in gene_expression_col_name_list:
    OS_feature_list = list(OS_gene_expression_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_gene_expression_transposed_df[col_name])

    # [OS/ nonOS]
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)
   

merged_transcriptomics_pvalue_df = pd.DataFrame({
    'gene_names': gene_expression_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_transcriptomics_pvalue_df)
merged_transcriptomics_pvalue_df.to_csv('./data_UCSC/stat_data/merged_transcriptomics_pvalue.csv', index=False, header=True)

## 6.4 Statistical analysis for epigenomics data

In [ ]:
### Keep the subject in the columns for certain types of patients for core promoters
# OS for core promoters
OS_merged_core_promoter_df = Core_Promoter_df_filtered[['gene_name'] + OS_list] # [68 t2ds]
OS_merged_core_promoter_transposed_df = OS_merged_core_promoter_df.T
OS_merged_core_promoter_transposed_df.columns = OS_merged_core_promoter_transposed_df.iloc[0]
OS_merged_core_promoter_transposed_df = OS_merged_core_promoter_transposed_df[1:]
OS_merged_core_promoter_transposed_df.reset_index(level=0, inplace=True)
OS_merged_core_promoter_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
# nonOs for core promoters
nonOS_merged_core_promoter_df = Core_Promoter_df_filtered[['gene_name'] + nonOS_list] # [105 pret2ds]
nonOS_merged_core_promoter_transposed_df = nonOS_merged_core_promoter_df.T
nonOS_merged_core_promoter_transposed_df.columns = nonOS_merged_core_promoter_transposed_df.iloc[0]
nonOS_merged_core_promoter_transposed_df = nonOS_merged_core_promoter_transposed_df[1:]
nonOS_merged_core_promoter_transposed_df.reset_index(level=0, inplace=True)
nonOS_merged_core_promoter_transposed_df.rename(columns={'index': 'subject'}, inplace=True)

from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

merged_core_promoter_col_name_list = list(OS_merged_core_promoter_transposed_df.columns)[1:]
for col_name in merged_core_promoter_col_name_list:
    OS_feature_list = list(OS_merged_core_promoter_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_merged_core_promoter_transposed_df[col_name])
    
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)

merged_core_promoter_pvalue_df = pd.DataFrame({
    'gene_names': merged_core_promoter_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_core_promoter_pvalue_df)
merged_core_promoter_pvalue_df.to_csv('./data_UCSC/stat_data/merged_core_promoter_pvalue.csv', index=False, header=True)

In [ ]:
### Keep the subject in the columns for certain types of patients for core promoters
# OS for core promoters
OS_merged_proximal_promoter_df = Proximal_Promoter_df_filtered[['gene_name'] + OS_list] # [68 t2ds]
OS_merged_proximal_promoter_transposed_df = OS_merged_proximal_promoter_df.T
OS_merged_proximal_promoter_transposed_df.columns = OS_merged_proximal_promoter_transposed_df.iloc[0]
OS_merged_proximal_promoter_transposed_df = OS_merged_proximal_promoter_transposed_df[1:]
OS_merged_proximal_promoter_transposed_df.reset_index(level=0, inplace=True)
OS_merged_proximal_promoter_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
# nonOs for core promoters
nonOS_merged_proximal_promoter_df = Proximal_Promoter_df_filtered[['gene_name'] + nonOS_list] # [105 pret2ds]
nonOS_merged_proximal_promoter_transposed_df = nonOS_merged_proximal_promoter_df.T
nonOS_merged_proximal_promoter_transposed_df.columns = nonOS_merged_proximal_promoter_transposed_df.iloc[0]
nonOS_merged_proximal_promoter_transposed_df = nonOS_merged_proximal_promoter_transposed_df[1:]
nonOS_merged_proximal_promoter_transposed_df.reset_index(level=0, inplace=True)
nonOS_merged_proximal_promoter_transposed_df.rename(columns={'index': 'subject'}, inplace=True)

from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

merged_proximal_promoter_col_name_list = list(OS_merged_proximal_promoter_transposed_df.columns)[1:]
for col_name in merged_proximal_promoter_col_name_list:
    OS_feature_list = list(OS_merged_proximal_promoter_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_merged_proximal_promoter_transposed_df[col_name])
    
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)

merged_proximal_promoter_pvalue_df = pd.DataFrame({
    'gene_names': merged_proximal_promoter_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_proximal_promoter_pvalue_df)
merged_proximal_promoter_pvalue_df.to_csv('./data_UCSC/stat_data/merged_proximal_promoter_pvalue.csv', index=False, header=True)

In [ ]:
### Keep the subject in the columns for certain types of patients for core promoters
# OS for core promoters
OS_merged_distal_promoter_df = Distal_Promoter_df_filtered[['gene_name'] + OS_list] # [68 t2ds]
OS_merged_distal_promoter_transposed_df = OS_merged_distal_promoter_df.T
OS_merged_distal_promoter_transposed_df.columns = OS_merged_distal_promoter_transposed_df.iloc[0]
OS_merged_distal_promoter_transposed_df = OS_merged_distal_promoter_transposed_df[1:]
OS_merged_distal_promoter_transposed_df.reset_index(level=0, inplace=True)
OS_merged_distal_promoter_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
# nonOs for core promoters
nonOS_merged_distal_promoter_df = Distal_Promoter_df_filtered[['gene_name'] + nonOS_list] # [105 pret2ds]
nonOS_merged_distal_promoter_transposed_df = nonOS_merged_distal_promoter_df.T
nonOS_merged_distal_promoter_transposed_df.columns = nonOS_merged_distal_promoter_transposed_df.iloc[0]
nonOS_merged_distal_promoter_transposed_df = nonOS_merged_distal_promoter_transposed_df[1:]
nonOS_merged_distal_promoter_transposed_df.reset_index(level=0, inplace=True)
nonOS_merged_distal_promoter_transposed_df.rename(columns={'index': 'subject'}, inplace=True)

from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

merged_distal_promoter_col_name_list = list(OS_merged_distal_promoter_transposed_df.columns)[1:]
for col_name in merged_distal_promoter_col_name_list:
    OS_feature_list = list(OS_merged_distal_promoter_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_merged_distal_promoter_transposed_df[col_name])
    
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)

merged_distal_promoter_pvalue_df = pd.DataFrame({
    'gene_names': merged_distal_promoter_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_distal_promoter_pvalue_df)
merged_distal_promoter_pvalue_df.to_csv('./data_UCSC/stat_data/merged_distal_promoter_pvalue.csv', index=False, header=True)

In [ ]:
### Keep the subject in the columns for certain types of patients for core promoters
# OS for core promoters
OS_merged_upstream_df = Upstream_df_filtered[['gene_name'] + OS_list] # [68 t2ds]
OS_merged_upstream_transposed_df = OS_merged_upstream_df.T
OS_merged_upstream_transposed_df.columns = OS_merged_upstream_transposed_df.iloc[0]
OS_merged_upstream_transposed_df = OS_merged_upstream_transposed_df[1:]
OS_merged_upstream_transposed_df.reset_index(level=0, inplace=True)
OS_merged_upstream_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
# nonOs for core promoters
nonOS_merged_upstream_df = Upstream_df_filtered[['gene_name'] + nonOS_list] # [105 pret2ds]
nonOS_merged_upstream_transposed_df = nonOS_merged_upstream_df.T
nonOS_merged_upstream_transposed_df.columns = nonOS_merged_upstream_transposed_df.iloc[0]
nonOS_merged_upstream_transposed_df = nonOS_merged_upstream_transposed_df[1:]
nonOS_merged_upstream_transposed_df.reset_index(level=0, inplace=True)
nonOS_merged_upstream_transposed_df.rename(columns={'index': 'subject'}, inplace=True)

from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

merged_upstream_col_name_list = list(OS_merged_upstream_transposed_df.columns)[1:]
for col_name in merged_upstream_col_name_list:
    OS_feature_list = list(OS_merged_upstream_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_merged_upstream_transposed_df[col_name])
    
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)

merged_upstream_pvalue_df = pd.DataFrame({
    'gene_names': merged_upstream_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_upstream_pvalue_df)
merged_upstream_pvalue_df.to_csv('./data_UCSC/stat_data/merged_upstream_pvalue.csv', index=False, header=True)

In [ ]:
### Keep the subject in the columns for certain types of patients for core promoters
# OS for core promoters
OS_merged_downstream_df = Downstream_df_filtered[['gene_name'] + OS_list] # [68 t2ds]
OS_merged_downstream_transposed_df = OS_merged_downstream_df.T
OS_merged_downstream_transposed_df.columns = OS_merged_downstream_transposed_df.iloc[0]
OS_merged_downstream_transposed_df = OS_merged_downstream_transposed_df[1:]
OS_merged_downstream_transposed_df.reset_index(level=0, inplace=True)
OS_merged_downstream_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
# nonOs for core promoters
nonOS_merged_downstream_df = Downstream_df_filtered[['gene_name'] + nonOS_list] # [105 pret2ds]
nonOS_merged_downstream_transposed_df = nonOS_merged_downstream_df.T
nonOS_merged_downstream_transposed_df.columns = nonOS_merged_downstream_transposed_df.iloc[0]
nonOS_merged_downstream_transposed_df = nonOS_merged_downstream_transposed_df[1:]
nonOS_merged_downstream_transposed_df.reset_index(level=0, inplace=True)
nonOS_merged_downstream_transposed_df.rename(columns={'index': 'subject'}, inplace=True)

from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

merged_downstream_col_name_list = list(OS_merged_downstream_transposed_df.columns)[1:]
for col_name in merged_downstream_col_name_list:
    OS_feature_list = list(OS_merged_downstream_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_merged_downstream_transposed_df[col_name])
    
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)

merged_downstream_pvalue_df = pd.DataFrame({
    'gene_names': merged_downstream_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_downstream_pvalue_df)
merged_downstream_pvalue_df.to_csv('./data_UCSC/stat_data/merged_downstream_pvalue.csv', index=False, header=True)

## 6.5 Statistical analysis for mutation data

In [ ]:
### Keep the subject in the columns for certain types of patients for core promoters
# OS for core promoters
OS_merged_mutation_df = mutation_filtered[['gene_name'] + OS_list] # [68 t2ds]
OS_merged_mutation_transposed_df = OS_merged_mutation_df.T
OS_merged_mutation_transposed_df.columns = OS_merged_mutation_transposed_df.iloc[0]
OS_merged_mutation_transposed_df = OS_merged_mutation_transposed_df[1:]
OS_merged_mutation_transposed_df.reset_index(level=0, inplace=True)
OS_merged_mutation_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
# nonOs for core promoters
nonOS_merged_mutation_df = mutation_filtered[['gene_name'] + nonOS_list] # [105 pret2ds]
nonOS_merged_mutation_transposed_df = nonOS_merged_mutation_df.T
nonOS_merged_mutation_transposed_df.columns = nonOS_merged_mutation_transposed_df.iloc[0]
nonOS_merged_mutation_transposed_df = nonOS_merged_mutation_transposed_df[1:]
nonOS_merged_mutation_transposed_df.reset_index(level=0, inplace=True)
nonOS_merged_mutation_transposed_df.rename(columns={'index': 'subject'}, inplace=True)

from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

merged_mutation_col_name_list = list(OS_merged_mutation_transposed_df.columns)[1:]
for col_name in merged_mutation_col_name_list:
    OS_feature_list = list(OS_merged_mutation_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_merged_mutation_transposed_df[col_name])
    
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)

merged_mutation_pvalue_df = pd.DataFrame({
    'gene_names': merged_mutation_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_mutation_pvalue_df)
merged_mutation_pvalue_df.to_csv('./data_UCSC/stat_data/merged_mutation_pvalue.csv', index=False, header=True)

## 6.6 Statistical analysis for protein data

In [ ]:
### Keep the subject in the columns for certain types of patients for core promoters
# OS for core promoters
OS_merged_protein_df = protein_filtered[['gene_name'] + OS_list] # [68 t2ds]
OS_merged_protein_transposed_df = OS_merged_protein_df.T
OS_merged_protein_transposed_df.columns = OS_merged_protein_transposed_df.iloc[0]
OS_merged_protein_transposed_df = OS_merged_protein_transposed_df[1:]
OS_merged_protein_transposed_df.reset_index(level=0, inplace=True)
OS_merged_protein_transposed_df.rename(columns={'index': 'subject'}, inplace=True)
# nonOs for core promoters
nonOS_merged_protein_df = protein_filtered[['gene_name'] + nonOS_list] # [105 pret2ds]
nonOS_merged_protein_transposed_df = nonOS_merged_protein_df.T
nonOS_merged_protein_transposed_df.columns = nonOS_merged_protein_transposed_df.iloc[0]
nonOS_merged_protein_transposed_df = nonOS_merged_protein_transposed_df[1:]
nonOS_merged_protein_transposed_df.reset_index(level=0, inplace=True)
nonOS_merged_protein_transposed_df.rename(columns={'index': 'subject'}, inplace=True)

from scipy.stats import ks_2samp
p_value_ks_OS_nonOS_list = []

merged_protein_col_name_list = list(OS_merged_protein_transposed_df.columns)[1:]
for col_name in merged_protein_col_name_list:
    OS_feature_list = list(OS_merged_protein_transposed_df[col_name])
    nonOS_feature_list = list(nonOS_merged_protein_transposed_df[col_name])
    
    ks_stat_OS_nonOS, p_value_ks_OS_nonOS = ks_2samp(OS_feature_list, nonOS_feature_list)
    p_value_ks_OS_nonOS_list.append(p_value_ks_OS_nonOS)

merged_protein_pvalue_df = pd.DataFrame({
    'gene_names': merged_protein_col_name_list,
    'OS_nonOS_pvalue': p_value_ks_OS_nonOS_list,
})
display(merged_protein_pvalue_df)
merged_protein_pvalue_df.to_csv('./data_UCSC/stat_data/merged_protein_pvalue.csv', index=False, header=True)